In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


catalog = "workspace"
silver_schema = f"{catalog}.olist_silver"
gold_schema = f"{catalog}.olist_gold"

# Read conformed source tables from the Silver schema
orders = spark.table(f"{silver_schema}.orders")
order_items = spark.table(f"{silver_schema}.order_items")
payments = spark.table(f"{silver_schema}.payments")
products = spark.table(f"{silver_schema}.products")
customers = spark.table(f"{silver_schema}.customers")
reviews = spark.table(f"{silver_schema}.reviews")

print("Gold Layer Business Marts Environment Initialized Successfully.")

In [0]:
print("Computing Mart 1: Revenue by Month...")

#how="inner" eliminates rows that do not find a matching key value on both sides
revenue_by_month = (
    orders.join(order_items, on="order_id", how="inner")
    .groupBy("purchase_year_month")
    .agg(F.sum("total_item_value").alias("monthly_gmv"))
    .orderBy("purchase_year_month")
)

revenue_by_month.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.revenue_by_month")
print("Saved table: gold.revenue_by_month")

In [0]:
print("Computing Mart 2: Top Product Categories...")

top_categories = (
    order_items.join(products, on="product_id", how="inner")
    .groupBy("product_category_name_english")
    .agg(
        F.sum("price").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders")
    )
    .orderBy(F.desc("total_revenue"))
)

top_categories.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.top_categories")
print("Saved table: gold.top_categories")